In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
def multi_col_apply(func):
    def wrapper(df, cols=None):
        if cols is None:
            cols = df.columns
        for c in cols:
            df = func(df, c)
        return df

    return wrapper


def apply_transformations(df, transformations):
    for func, kwargs in transformations:
        df = func(df, **kwargs)
    return df

In [0]:
@multi_col_apply
def trimming(df, col):
    return df.withColumn(col, F.trim(F.col(col)))


@multi_col_apply
def uppercase(df, col):
    return df.withColumn(col, F.upper(F.col(col)))


@multi_col_apply
def lowercase(df, col):
    return df.withColumn(col, F.lower(F.col(col)))


@multi_col_apply
def capitalize(df, col):
    return df.withColumn(col, F.initcap(F.col(col)))


def mobile_num_cleanup(df, col):
    new_col_name = col + "_cleaned"
    return (
        df.withColumn(
            new_col_name,
            F.regexp_replace(F.col(col), r"^\+\d{2}[\s\-\(\)]*|[\s\-\(\)]", ""),
        )
        .withColumn(new_col_name, F.substring(new_col_name, -10, 10))
        .withColumn(
            f"valid_{col}",
            F.when(F.length(F.col(new_col_name)) == 10, True).otherwise(False),
        )
    )


def handle_duplicate(df, col1, col2):
    return (
        df.withColumn(
            "rn",
            F.rank().over(
                Window.partitionBy(F.col(f"{col1}")).orderBy(F.col(f"{col2}"))
            ),
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

In [0]:
def unify_customers_data(df_offline_customers, df_online_customers):
    online_cust_transformations_1 = [
        (mobile_num_cleanup, {"col": "mobile_number"}),
        (trimming, {}),
        (capitalize, {"cols": ["full_name", "city"]}),
        (lowercase, {"cols": ["email"]}),
        (uppercase, {"cols": ["customer_id", "state"]}),
    ]
    df_online_transformed_1 = apply_transformations(
        df_online_customers, online_cust_transformations_1
    )
    df_online_cust_quarantine = (
        df_online_transformed_1.filter(F.col("valid_mobile_number") == False)
        .withColumn("valid_mobile_number", F.col("valid_mobile_number").cast("boolean"))
        .withColumn("quarantine_reason", F.lit("Invalid mobile number"))
        .withColumn("quarantine_ts", F.lit(datetime.now()))
    )
    df_online_cust_quarantine.write.mode("append").saveAsTable(
        "retail_data.quarantine.online_customer_quarantine"
    )

    online_cust_transformations_2 = [
        (handle_duplicate, {"col1": "mobile_number_cleaned", "col2": "customer_id"})
    ]

    df_online_transformed = apply_transformations(
        df_online_transformed_1.filter(F.col("valid_mobile_number") == True),
        online_cust_transformations_2,
    )

    offline_cust_transformations_1 = [
        (mobile_num_cleanup, {"col": "phone"}),
        (trimming, {}),
        (capitalize, {"cols": ["name", "city"]}),
        (lowercase, {"cols": ["email"]}),
        (uppercase, {"cols": ["cust_id", "phone"]}),
    ]

    df_offline_transformed_1 = apply_transformations(
        df_offline_customers, offline_cust_transformations_1
    )

    df_offline_cust_quarantine = (
        df_offline_transformed_1.filter(F.col("valid_phone") == False)
        .withColumn("valid_phone", F.col("valid_phone").cast("boolean"))
        .withColumn("quarantine_reason", F.lit("Invalid mobile number"))
        .withColumn("quarantine_ts", F.lit(datetime.now()))
    )
    df_offline_cust_quarantine.write.mode("append").saveAsTable(
        "retail_data.quarantine.offline_customer_quarantine"
    )

    offline_cust_transformations_2 = [
        (handle_duplicate, {"col1": "phone_cleaned", "col2": "cust_id"})
    ]

    df_offline_transformed = apply_transformations(
        df_offline_transformed_1.filter(F.col("valid_phone") == True),
        offline_cust_transformations_2,
    )

    df_common_customers = df_offline_transformed.join(
        df_online_transformed,
        df_offline_transformed["phone_cleaned"]
        == df_online_transformed["mobile_number_cleaned"],
        "inner",
    ).select(
        F.coalesce(F.col("full_name"), F.col("name")).cast("string").alias("cust_name"),
        F.coalesce(df_online_transformed["email"], df_offline_transformed["email"])
        .cast("string")
        .alias("cust_email"),
        F.col("mobile_number_cleaned").cast("string").alias("cust_mobile"),
        F.coalesce(df_online_transformed["city"], df_offline_transformed["city"])
        .cast("string")
        .alias("cust_city"),
        F.coalesce(df_online_transformed["state"], df_offline_transformed["state"])
        .cast("string")
        .alias("cust_state"),
    )

    df_online_only_customers = df_online_transformed.join(
        df_offline_transformed,
        df_online_transformed["mobile_number_cleaned"]
        == df_offline_transformed["phone_cleaned"],
        "left_anti",
    ).select(
        F.col("full_name").cast("string").alias("cust_name"),
        df_online_transformed["email"].cast("string").alias("cust_email"),
        F.col("mobile_number_cleaned").cast("string").alias("cust_mobile"),
        df_online_transformed["city"].cast("string").alias("cust_city"),
        df_online_transformed["state"].cast("string").alias("cust_state"),
    )

    df_offline_only_customers = df_offline_transformed.join(
        df_online_transformed,
        df_online_transformed["mobile_number_cleaned"]
        == df_offline_transformed["phone_cleaned"],
        "left_anti",
    ).select(
        F.col("name").cast("string").alias("cust_name"),
        df_offline_transformed["email"].cast("string").alias("cust_email"),
        F.col("phone_cleaned").cast("string").alias("cust_mobile"),
        df_offline_transformed["city"].cast("string").alias("cust_city"),
        df_offline_transformed["state"].cast("string").alias("cust_state"),
    )
    df_customers_unified = df_common_customers.union(df_online_only_customers).union(
        df_offline_only_customers
    )
    return df_customers_unified

In [0]:
def unify_products_data(df_offline_products, df_online_products):
    online_products_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["product_name", "category", "brand"]}),
        (uppercase, {"cols": ["sku_id"]}),
    ]

    df_online_transformed = apply_transformations(
        df_online_products, online_products_transformations
    )

    offline_products_transformations = [
        (trimming, {}),
        (capitalize, {"cols": ["name", "category_name", "brand_name"]}),
        (uppercase, {"cols": ["sku"]}),
    ]

    df_offline_transformed = apply_transformations(
        df_offline_products, offline_products_transformations
    )

    df_common_products = df_offline_transformed.join(
        df_online_transformed,
        df_offline_transformed["sku"] == df_online_transformed["sku_id"],
        "inner",
    ).select(
        F.col("sku_id").cast("string").alias("product_id"),
        F.coalesce(F.col("product_name"), F.col("name"))
        .cast("string")
        .alias("product_name"),
        F.coalesce(F.col("category"), F.col("category_name"))
        .cast("string")
        .alias("product_category"),
        F.coalesce(F.col("brand"), F.col("brand_name"))
        .cast("string")
        .alias("product_brand"),
        F.coalesce(F.col("price"), F.col("mrp_price"))
        .cast("float")
        .alias("product_price"),
    )

    df_online_only_products = df_online_transformed.join(
        df_offline_transformed,
        df_online_transformed["sku_id"] == df_offline_transformed["sku"],
        "left_anti",
    ).select(
        F.col("sku_id").cast("string").alias("product_id"),
        F.col("product_name").cast("string").alias("product_name"),
        F.col("category").cast("string").alias("product_category"),
        F.col("brand").cast("string").alias("product_brand"),
        F.col("price").cast("float").alias("product_price"),
    )

    df_offline_only_products = df_offline_transformed.join(
        df_online_transformed,
        df_online_transformed["sku_id"] == df_offline_transformed["sku"],
        "left_anti",
    ).select(
        F.col("sku").cast("string").alias("product_id"),
        F.col("name").cast("string").alias("product_name"),
        F.col("category_name").cast("string").alias("product_category"),
        F.col("brand_name").cast("string").alias("product_brand"),
        F.col("mrp_price").cast("float").alias("product_price"),
    )

    df_products_unified = df_common_products.union(df_online_only_products).union(
        df_offline_only_products
    )

    return df_products_unified

In [0]:
def unify_orders_data(df_offline_orders, df_online_orders, unified_products):
    df_online_orders_with_phone = df_online_orders.join(
        df_online_customers, "customer_id", "left"
    ).select(df_online_orders["*"], df_online_customers["mobile_number"])

    online_orders_transformations = [
        (mobile_num_cleanup, {"col": "mobile_number"}),
    ]
    df_online_transformed = apply_transformations(
        df_online_orders_with_phone, online_orders_transformations
    )

    df_online_orders_quarantine_1 = (
        df_online_transformed.filter(F.col("valid_mobile_number") == False)
        .withColumn("valid_mobile_number", F.col("valid_mobile_number").cast("boolean"))
        .withColumn("quarantine_reason", F.lit("Invalid mobile number"))
        .withColumn("quarantine_ts", F.lit(datetime.now()))
    )

    df_online_orders_quarantine_2 = (
        df_online_transformed.join(
            df_online_orders_quarantine_1.select("order_id").distinct(),
            on="order_id",
            how="left_anti",
        )
        .join(
            unified_customers,
            df_online_transformed.mobile_number_cleaned
            == unified_customers.cust_mobile,
            "left_anti",
        )
        .withColumn("valid_mobile_number", F.col("valid_mobile_number").cast("boolean"))
        .withColumn("quarantine_reason", F.lit("Customer lookup failed"))
        .withColumn("quarantine_ts", F.lit(datetime.now()))
    )

    df_online_orders_quarantine = df_online_orders_quarantine_1.union(
        df_online_orders_quarantine_2
    )
    df_online_orders_quarantine.write.mode("append").saveAsTable(
        "retail_data.quarantine.online_orders_quarantine"
    )

    df_online_orders_final = (
        df_online_transformed.filter(
            (F.col("valid_mobile_number") == True)
            & (df_online_transformed.mobile_number_cleaned.isNotNull())
        )
        .join(
            unified_customers,
            df_online_transformed.mobile_number_cleaned
            == unified_customers.cust_mobile,
            "inner",
        )
        .withColumn("list_price", F.col("list_price").cast("double"))
        .withColumn("qty", F.col("qty").cast("int"))
        .withColumn(
            "discount", F.coalesce(F.col("discount"), F.lit(0.0)).cast("double")
        )
        .withColumn("amount", (F.col("list_price") * F.col("qty") - F.col("discount")))
        .withColumn("amount", F.col("amount").cast("double"))
        .select(
            F.col("order_id").cast("string").alias("order_id"),
            F.col("order_ts").cast("timestamp").alias("order_ts"),
            F.col("source").cast("string").alias("source_system"),
            F.col("customer_id").cast("string").alias("customer_id"),
            F.col("sku_id").cast("string").alias("sku_id"),
            F.col("payment_mode").cast("string").alias("payment_mode"),
            F.col("qty").alias("quantity"),
            F.col("list_price"),
            F.col("discount"),
            F.col("amount"),
            F.lit("ONLINE").alias("store_num"),
        )
    )

    df_offline_orders_with_phone = df_offline_orders.join(
        df_offline_customers,
        df_offline_customers.cust_id == df_offline_orders.customer_id,
        "left",
    ).select(df_offline_orders["*"], df_offline_customers["phone"])

    offline_orders_transformations = [
        (mobile_num_cleanup, {"col": "phone"}),
    ]

    df_offline_transformed = apply_transformations(
        df_offline_orders_with_phone, offline_orders_transformations
    )

    df_offline_orders_quarantine_1 = (
        df_offline_transformed.filter(F.col("valid_phone") == False)
        .withColumn("valid_phone", F.col("valid_phone").cast("boolean"))
        .withColumn("quarantine_reason", F.lit("Invalid mobile number"))
        .withColumn("quarantine_ts", F.lit(datetime.now()))
    )

    df_offline_orders_quarantine_2 = (
        df_offline_transformed.join(
            df_offline_orders_quarantine_1.select("txn_id").distinct(),
            on="txn_id",
            how="left_anti",
        )
        .join(
            unified_customers,
            df_offline_transformed.phone_cleaned == unified_customers.cust_mobile,
            "left_anti",
        )
        .withColumn("valid_phone", F.col("valid_phone").cast("boolean"))
        .withColumn("quarantine_reason", F.lit("Customer lookup failed"))
        .withColumn("quarantine_ts", F.lit(datetime.now()))
    )

    df_offline_orders_quarantine = df_offline_orders_quarantine_1.union(
        df_offline_orders_quarantine_2
    )
    df_offline_orders_quarantine.write.mode("append").saveAsTable(
        "retail_data.quarantine.offline_orders_quarantine"
    )

    df_offline_orders_final = (
        df_offline_transformed.filter(
            (F.col("valid_phone") == True)
            & (df_offline_transformed.phone_cleaned.isNotNull())
        )
        .join(
            unified_customers,
            df_offline_transformed.phone_cleaned == unified_customers.cust_mobile,
            "inner",
        )
        .withColumn("mrp", F.col("mrp").cast("double"))
        .withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn(
            "discount", F.coalesce(F.col("discount"), F.lit(0.0)).cast("double")
        )
        .withColumn("amount", (F.col("mrp") * F.col("quantity") - F.col("discount")))
        .withColumn("amount", F.col("amount").cast("double"))
        .select(
            F.col("txn_id").cast("string").alias("order_id"),
            F.col("txn_time").cast("timestamp").alias("order_ts"),
            F.col("source").cast("string").alias("source_system"),
            F.col("customer_id").cast("string").alias("customer_id"),
            F.col("sku").cast("string").alias("sku_id"),
            F.col("payment_mode").cast("string").alias("payment_mode"),
            F.col("quantity"),
            F.col("mrp").alias("list_price"),
            F.col("discount"),
            F.col("amount"),
            F.col("store_num"),
        )
    )
    df_orders_unified = df_online_orders_final.union(df_offline_orders_final)
    return df_orders_unified

In [0]:
df_offline_customers = spark.table("retail_data.bronze.offline_customers")
df_online_customers = spark.table("retail_data.bronze.online_customers")
df_offline_products = spark.table("retail_data.bronze.offline_products")
df_online_products = spark.table("retail_data.bronze.online_products")
df_offline_orders = spark.table("retail_data.bronze.offline_orders")
df_online_orders = spark.table("retail_data.bronze.online_orders")

In [0]:
unified_customers = unify_customers_data(df_offline_customers, df_online_customers)
unified_customers.write.mode("overwrite").saveAsTable(
    "retail_data.silver.customers_unified"
)

In [0]:
unified_products = unify_products_data(df_offline_products, df_online_products)
unified_products.write.mode("overwrite").saveAsTable(
    "retail_data.silver.products_unified"
)

In [0]:
unified_orders = unify_orders_data(df_offline_orders, df_online_orders, unified_products)
unified_orders.write.mode("overwrite").saveAsTable(
    "retail_data.silver.orders_unified"
)